# Sequence Labeling — BIO tagging and why token accuracy lies

Named entity recognition as token classification, and the evaluation mistake that makes a
useless model look excellent.

**Runs offline in seconds.** Theory: [`02_Tasks_and_Evaluation.md`](02_Tasks_and_Evaluation.md) §3.

In [ ]:
import numpy as np
from collections import Counter
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression


In [ ]:
SYL_A = "ka me lo ri sa tu vi na do pe mi ra ko lu se ti va ne".split()
SYL_B = "ndra lson vich ami rez oka tani berg wski sson mura hani".split()
SUFFIX_ORG = ["Corp", "Group", "Labs", "Systems", "Holdings"]
FILLER = """the a of and to in for with on at by from said later after before during while
report meeting project team year month data system value result group plan review note
issue update version change request service process record case study""".split()
TYPES = ["PER", "LOC", "ORG"]


## 1. Data with *unseen* entity names

Names are generated from syllables, and train/test use different seeds — so most test
entities were never seen in training. That matters: with a fixed list of names the model just
memorizes them and scores a meaningless 100%.

Sentence-initial words are randomly capitalized, which is a real source of NER false
positives.

In [ ]:
def gen_name(rng, n_syl=2):
    return "".join(rng.choice(SYL_A) for _ in range(n_syl)).capitalize()


def gen_entity(rng, typ):
    if typ == "PER":
        return [gen_name(rng), gen_name(rng) + str(rng.choice(SYL_B))]
    if typ == "LOC":
        return [gen_name(rng)] + ([gen_name(rng)] if rng.random() < 0.4 else [])
    toks = [gen_name(rng), str(rng.choice(SUFFIX_ORG))]
    if rng.random() < 0.3:
        toks.insert(1, gen_name(rng))
    return toks


In [ ]:
def make_sentences(seed, n=1200, filler_run=(4, 12), ents=(1, 3)):
    """Names are generated, so a different seed gives UNSEEN names at test time.
    That forces the model onto shape/context features instead of memorizing a list."""
    rng = np.random.default_rng(seed)
    sents = []
    for _ in range(n):
        toks, tags = [], []

        def fill():
            for _ in range(rng.integers(*filler_run)):
                w = str(rng.choice(FILLER))
                if not toks and rng.random() < 0.5:
                    w = w.capitalize()      # sentence-initial caps: a real NER distractor
                toks.append(w)
                tags.append("O")

        fill()
        for _ in range(rng.integers(*ents)):
            typ = str(rng.choice(TYPES))
            e = gen_entity(rng, typ)
            toks.extend(e)
            tags.extend([f"B-{typ}"] + [f"I-{typ}"] * (len(e) - 1))
            fill()
        sents.append((toks, tags))
    return sents


In [ ]:
train_sents = make_sentences(seed=0, n=1200)
test_sents = make_sentences(seed=99, n=400)
print(f"sentences: train {len(train_sents)}  test {len(test_sents)}")
toks, tags = train_sents[0]
print("\nexample (BIO tagging):")
for a, b in list(zip(toks, tags))[:14]:
    print(f"   {a:14s} {b}")


## 2. Features

No embeddings — just the classical feature template: the word, its **shape** (`Xxxx`), affixes,
and the immediate neighbours. Shape and context are what let the model tag a name it has
never seen.

In [ ]:
dist = Counter(t for _, ts in train_sents for t in ts)
print("\ntag distribution:", dict(dist.most_common()))
print(f"  share of 'O': {dist['O']/sum(dist.values()):.1%}")

train_names = {w for t, g in train_sents for w, gg in zip(t, g) if gg != "O"}
test_names = {w for t, g in test_sents for w, gg in zip(t, g) if gg != "O"}
print(f"  entity tokens unseen in training: {len(test_names - train_names)/len(test_names):.1%}")


In [ ]:
def shape(w):
    s = "".join("X" if c.isupper() else "x" if c.islower() else "d" if c.isdigit() else "-" for c in w)
    out = [s[0]]
    for c in s[1:]:
        if c != out[-1]:
            out.append(c)
    return "".join(out)


def feats(toks, i):
    w = toks[i]
    f = {
        "bias": 1.0,
        "w.lower": w.lower(),
        "w.istitle": w.istitle(),
        "w.shape": shape(w),
        "w.suffix3": w[-3:].lower(),
        "w.prefix2": w[:2].lower(),
        "pos.first": i == 0,
    }
    f["-1.w"] = toks[i-1].lower() if i > 0 else "<BOS>"
    f["-1.istitle"] = toks[i-1].istitle() if i > 0 else False
    f["+1.w"] = toks[i+1].lower() if i < len(toks)-1 else "<EOS>"
    f["+1.istitle"] = toks[i+1].istitle() if i < len(toks)-1 else False
    return f


In [ ]:
Xtr_d = [feats(t, i) for t, _ in train_sents for i in range(len(t))]
ytr = [g for _, gs in train_sents for g in gs]
Xte_d = [feats(t, i) for t, _ in test_sents for i in range(len(t))]
yte = [g for _, gs in test_sents for g in gs]
dv = DictVectorizer()
Xtr, Xte = dv.fit_transform(Xtr_d), dv.transform(Xte_d)
print(f"\ntokens: train {len(ytr)}  test {len(yte)}  | features {Xtr.shape[1]}")


## 3. The trap

Around 87% of tokens are `O`. So a model that predicts `O` for everything — finding zero
entities — already scores ~87% token accuracy.

In [ ]:
clf = LogisticRegression(max_iter=1000, C=5).fit(Xtr, ytr)
pred = list(clf.predict(Xte))


## 4. Entity-level scoring

The real metric. Decode BIO tags into `(start, end, type)` spans and require an **exact**
match on all three. No partial credit: predicting `B-PER O` for a two-token person is one
false positive *and* one false negative, even though token accuracy for that span is 50%.

This is what `seqeval` and the CoNLL scorer compute.

In [ ]:
tok_acc = np.mean([p == t for p, t in zip(pred, yte)])
all_O = np.mean([t == "O" for t in yte])
print(f"\n--- the token-accuracy trap ---")
print(f"  token accuracy          {tok_acc:.4f}")
print(f"  all-'O' baseline        {all_O:.4f}   <- predicting nothing scores this")


In [ ]:
def spans(tags):
    """BIO -> set of (start, end, type). An I-X not continuing an X starts a new span."""
    out, start, typ = [], None, None
    for i, t in enumerate(list(tags) + ["O"]):
        if t.startswith("B-"):
            if typ:
                out.append((start, i, typ))
            start, typ = i, t[2:]
        elif t.startswith("I-"):
            if typ != t[2:]:
                if typ:
                    out.append((start, i, typ))
                start, typ = i, t[2:]
        else:
            if typ:
                out.append((start, i, typ))
            start, typ = None, None
    return set(out)


In [ ]:
def entity_prf(gold, pred_):
    tp = fp = fn = 0
    for g, p in zip(gold, pred_):
        G, P = spans(g), spans(p)
        tp += len(G & P); fp += len(P - G); fn += len(G - P)
    pr = tp/(tp+fp) if tp+fp else 0.0
    rc = tp/(tp+fn) if tp+fn else 0.0
    return pr, rc, (2*pr*rc/(pr+rc) if pr+rc else 0.0), tp, fp, fn


In [ ]:
gold_sents, pred_sents, k = [], [], 0
for toks, gs in test_sents:
    n = len(toks)
    gold_sents.append(gs); pred_sents.append(pred[k:k+n]); k += n

pr, rc, f1, tp, fp, fn = entity_prf(gold_sents, pred_sents)
print(f"\n--- entity-level (exact span AND type) ---")
print(f"  precision {pr:.3f}   recall {rc:.3f}   F1 {f1:.3f}    (tp={tp} fp={fp} fn={fn})")
print(f"  token accuracy said {tok_acc:.3f}; entity F1 is {f1:.3f}. That gap is the point.")


## 5. Structural validity — the case for a CRF

`I-PER` cannot legally follow `O` or `B-LOC`. A per-token classifier decides each tag
independently, so nothing stops it emitting impossible sequences.

A **linear-chain CRF** scores the whole sequence with a learned transition matrix, which gives
illegal transitions an effectively infinite penalty, and decodes with Viterbi.

In [ ]:
print("\nper-type:")
for typ in TYPES:
    g2 = [[t if t.endswith(typ) else "O" for t in s] for s in gold_sents]
    p2 = [[t if t.endswith(typ) else "O" for t in s] for s in pred_sents]
    a, b, c, *_ = entity_prf(g2, p2)
    print(f"  {typ}: P {a:.3f}  R {b:.3f}  F1 {c:.3f}")


## 6. A concrete error

In [ ]:
illegal = 0
for s in pred_sents:
    prev = "O"
    for t in s:
        if t.startswith("I-") and not (prev != "O" and prev.endswith(t[2:])):
            illegal += 1
        prev = t
print(f"\n--- structural validity ---")
print(f"  invalid transitions emitted: {illegal}")
print("  An I-X that doesn't follow B-X/I-X is impossible in valid BIO. A per-token")
print("  classifier cannot forbid it; a CRF's learned transition matrix is exactly")
print("  what makes those sequences score -inf.")


## Takeaways

1. **Never report token accuracy for NER.** With ~87% `O`, predicting nothing scores ~87%.
   Use entity-level precision/recall/F1 with exact span+type match.
2. **The gap is enormous** — ~0.98 token accuracy against ~0.69 entity F1 here, on the same
   predictions.
3. **Exact match is unforgiving**, and deliberately so: a half-captured entity is usually
   useless downstream.
4. **Per-token classification emits invalid structure.** A CRF, or post-hoc repair, is what
   fixes it.
5. **Generalizing to unseen names** is the actual task — which is why shape and context
   features carry the model, and why memorizable data gives misleading results.

**Try next:** add a gazetteer feature (is this word in a known name list?) and watch precision
move; or implement Viterbi decoding over the classifier's probabilities with a hand-written
transition matrix that forbids illegal moves, and re-score.